In [ ]:
# KALMAN OPEN POLICY CONTRACT DISCOVERY v2.5 — ONE CELL / READ ONLY
# Goal: recover frozen OPEN_* trigger/exit/net-return semantics from the legacy 20 rows.
from google.colab import drive
drive.mount("/content/drive",force_remount=False)
from pathlib import Path
import numpy as np,pandas as pd,json,re

R=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results")
A=pd.read_parquet(R/"open_revalidation_v1/open_revalidation_trade_audit.parquet")
print("[AUDIT]",A.shape)
ready=A["reconstructed_fixed4_net_return"].notna()
legacy=A["OPEN_NEG_0BP_0M__net_return"].notna() if "OPEN_NEG_0BP_0M__net_return" in A else pd.Series(False,index=A.index)
print("[DERIVED READY]",int(ready.sum()),"[LEGACY OPEN ROWS]",int(legacy.sum()))

base=["fold","symbol","weight","cost_proxy","position_return_prev_close","position_return_open","position_return_5m","position_return_15m","overnight_gap_return","open_momentum_5m","open_momentum_15m","giveback_prev_close_to_5m","reconstructed_fixed4_net_return"]
policies=sorted({c.split("__")[0] for c in A.columns if c.startswith("OPEN_")})
print("[POLICIES]",policies)

# Summarize legacy trigger behavior and exact exit/net formulas.
rows=[]
for p in policies:
 tc=p+"__trigger"; nc=p+"__net_return"; ec=p+"__exit_price"
 sub=A.loc[legacy,base+[tc,nc,ec]].copy()
 trig=sub[tc].fillna(False).astype(bool)
 rows.append({"policy":p,"legacy_rows":len(sub),"trigger_true":int(trig.sum()),"trigger_false":int((~trig).sum()),
              "net_na":int(sub[nc].isna().sum()),"exit_price_trigger_na":int(sub.loc[trig,ec].isna().sum())})
print("\n[LEGACY POLICY SUMMARY]\n",pd.DataFrame(rows).to_string(index=False))

# Candidate trigger predicates implied by names. Test exact agreement only; do not write.
pred={}
x=A.loc[legacy].copy()
pred["OPEN_NEG_0BP_0M"] = x["position_return_open"] < 0
pred["OPEN_NEG_20BP_0M"] = x["position_return_open"] < -0.002
# FLIP candidates: prior position sign positive -> open sign negative
pred["OPEN_FLIP_0M"] = (x["position_return_prev_close"] >= 0) & (x["position_return_open"] < 0)
pred["OPEN_NEG_5M_CONFIRM"] = (x["position_return_open"] < 0) & (x["position_return_5m"] < 0)
pred["OPEN_FLIP_5M_CONFIRM"] = (x["position_return_prev_close"] >= 0) & (x["position_return_open"] < 0) & (x["position_return_5m"] < 0)
# gap/giveback candidate variants are printed, not assumed
pred["OPEN_GAP_5M_CONFIRM"] = (x["overnight_gap_return"] < 0) & (x["position_return_5m"] < 0)
pred["OPEN_GIVEBACK_5M"] = x["giveback_prev_close_to_5m"] > 0
pred["OPEN_NEG_15M_CONFIRM"] = (x["position_return_open"] < 0) & (x["position_return_15m"] < 0)

print("\n[TRIGGER CANDIDATE AGREEMENT]")
for p in policies:
 actual=x[p+"__trigger"].fillna(False).astype(bool)
 if p in pred:
  cand=pred[p].fillna(False).astype(bool)
  print(p,"agree=",int((actual==cand).sum()),"/",len(x),"actual_true=",int(actual.sum()),"candidate_true=",int(cand.sum()))
  if not (actual==cand).all():
   show=x.loc[actual!=cand,base+[p+"__trigger"]].copy()
   show["candidate"]=cand.loc[actual!=cand]
   print(show.to_string(index=False))

# For triggered legacy rows, test whether exit price maps to open0/open5/open15 and
# whether net = position-return-at-exit * weight - cost_proxy.
print("\n[EXIT/NET CONTRACT TEST]")
for p in policies:
 trig=x[p+"__trigger"].fillna(False).astype(bool)
 if not trig.any():
  print(p,"no legacy triggers"); continue
 ep=pd.to_numeric(x.loc[trig,p+"__exit_price"],errors="coerce")
 candidates={"open0":"open_0_price_iex","open5":"open_5_price_iex","open15":"open_15_price_iex","fixed4":"fixed4_exit_price_iex"}
 print("\n",p,"n=",int(trig.sum()))
 for nm,col in candidates.items():
  if col not in x: continue
  px=pd.to_numeric(x.loc[trig,col],errors="coerce")
  print(" exit=",nm,"price_max_abs_err=",float((ep-px).abs().max()) if len(ep) else None)
 nr=pd.to_numeric(x.loc[trig,p+"__net_return"],errors="coerce")
 for nm,retcol in [("open0","position_return_open"),("open5","position_return_5m"),("open15","position_return_15m")]:
  rr=pd.to_numeric(x.loc[trig,retcol],errors="coerce")*pd.to_numeric(x.loc[trig,"weight"],errors="coerce")-pd.to_numeric(x.loc[trig,"cost_proxy"],errors="coerce")
  print(" net=",nm+"*weight-cost","max_abs_err=",float((nr-rr).abs().max()))

# Non-trigger behavior
print("\n[NON-TRIGGER NET CONTRACT]")
for p in policies:
 actual=x[p+"__trigger"].fillna(False).astype(bool)
 nr=pd.to_numeric(x.loc[~actual,p+"__net_return"],errors="coerce")
 base_net=pd.to_numeric(x.loc[~actual,"reconstructed_fixed4_net_return"],errors="coerce")
 print(p,"n=",int((~actual).sum()),"vs_fixed4_max_abs_err=",float((nr-base_net).abs().max()) if len(nr) else None)

print("\nREAD ONLY. No canonical file modified.")
print("NEXT: only policies whose trigger + exit + net contracts reproduce legacy rows exactly will be expanded to 888 rows.")
